In [ ]:
## 1. Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings("ignore")

In [ ]:
## 2. Load Dataset
train = pd.read_csv("/kaggle/input/adiip-2026-programming-test-no-1-ml-predict/train.csv")

print("Train shape:", train.shape)
train.head()

In [ ]:
test = pd.read_csv("/kaggle/input/adiip-2026-programming-test-no-1-ml-predict/test.csv")

print("Test shape:", test.shape)
test.head()

In [ ]:
# Target Distribution
train['Y'].value_counts()
train['Y'].value_counts(normalize=True)

In [ ]:
## 3. Exploratory Data Analysis (EDA)
# 3.1 Basic Info
train.info()

The dataset contains 24,000 observations with 25 variables. All features are numeric and no missing values were found, indicating that the dataset is clean and ready for modeling.

In [ ]:
# 3.2 Target Distribution
train['Y'].value_counts(normalize=True)

# Visualization
sns.countplot(x='Y', data=train)
plt.title("Target Distribution")
plt.show()

The dataset shows class imbalance where non-default customers dominate the dataset. Approximately 22% of customers experienced default. Therefore, ROC-AUC will be used as the primary evaluation metric.

In [ ]:
# 3.3 Correlation Check
plt.figure(figsize=(12,8))
corr = train.corr()
sns.heatmap(corr[['Y']].sort_values(by='Y', ascending=False), annot=True, cmap='coolwarm')
plt.title("Correlation with Target")
plt.show()

Payment history variables (X6–X11) show the strongest positive correlation with default. This indicates that past payment behavior is a strong predictor of future default risk.

In [ ]:
## 4. Data Preprocessing
# Separate features and target
X = train.drop(columns=['Y'])
y = train['Y']

# Drop ID (not useful for modeling)
X = X.drop(columns=['ID'])
test_id = test['ID']
test = test.drop(columns=['ID'])

print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
## 5. Modeling
# 5.1 Train–Validation Split
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Validation shape:", X_val.shape)

In [ ]:
# 5.2 Random Forest
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

val_pred_proba = rf.predict_proba(X_val)[:, 1]
roc_auc = roc_auc_score(y_val, val_pred_proba)

print("Validation ROC-AUC:", roc_auc)

In [ ]:
## 6. Model Evaluation
from sklearn.metrics import roc_curve

fpr, tpr, _ = roc_curve(y_val, val_pred_proba)

plt.figure(figsize=(6,5))
plt.plot(fpr, tpr)
plt.plot([0,1], [0,1])
plt.title("ROC Curve")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.show()

The Random Forest model achieved a validation ROC-AUC score of 0.756, indicating good discriminative ability in predicting default risk.

In [ ]:
# Retrain on full training data
rf_final = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

rf_final.fit(X, y)

# Predict probability for test set
test_pred = rf_final.predict(test)

# Create submission dataframe
submission = pd.DataFrame({
    "ID": test_id,
    "Y": test_pred
})

submission.to_csv("Khansa_Dzakiyyah-programming_1-ADIIP2026A.csv", index=False)

In [ ]:
submission.head()